# 🌍 Exploratory Data Analysis (EDA): World Development Indicators (WDI)

## 1. Giới thiệu tổng quan về Dataset
Bộ dữ liệu **World Development Indicators (WDI)** là bộ sưu tập dữ liệu thống kê hàng đầu của Ngân hàng Thế giới (World Bank) về sự phát triển toàn cầu. File Excel đầu vào chứa 6 sheets, bao gồm 3 sheet dữ liệu/metadata cốt lõi và 3 sheet phụ trợ.

Dưới đây là giải phẫu chi tiết ý nghĩa của từng sheet và các trường dữ liệu (fields):

### 1.1. Các sheet dữ liệu cốt lõi (Core Sheets)

**👉 Sheet `Data` (Bảng dữ liệu chính - 70 trường):**
Đây là nơi chứa toàn bộ các con số thống kê thực tế, được trình bày dưới dạng "Wide-format" (Định dạng rộng - mỗi năm là một cột).
* **Các trường định danh:** `Country Name` (Tên quốc gia/Khu vực), `Country Code` (Mã quốc gia chuẩn ISO 3-alpha, ví dụ: VNM), `Indicator Name` (Tên chỉ số đo lường), `Indicator Code` (Mã định danh duy nhất của chỉ số).
* **Các trường thời gian:** Các cột từ `1960` đến `2025` chứa giá trị thực tế của chỉ số tương ứng với quốc gia và năm đó.

**👉 Sheet `Country` (Từ điển Quốc gia - 31 trường):**
Cung cấp "bối cảnh" phân loại địa lý, kinh tế và phương pháp thống kê của 265 quốc gia/vùng lãnh thổ.
* **Nhóm Phân loại trọng yếu:** `Country Code` (Khóa ngoại để liên kết với sheet Data), `Region` (Khu vực địa lý - *rất quan trọng để phân tích theo châu lục*), `Income Group` (Nhóm thu nhập do WB phân loại: Cao, Trung bình cao, Trung bình thấp, Thấp).
* **Nhóm Thông tin chung:** `Short Name`, `Table Name`, `Long Name`, `2-alpha code` (Mã quốc gia 2 chữ cái), `Currency Unit` (Đơn vị tiền tệ quốc gia).
* **Nhóm Kế toán & Thống kê quốc gia:** `System of National Accounts`, `National accounts base year`, `SNA price valuation` (Chỉ ra hệ thống và năm cơ sở mà quốc gia dùng để tính toán GDP/GNP).
* **Nhóm Điều tra dân số & Khảo sát:** `Latest population census` (Năm điều tra dân số gần nhất), `Latest household survey`, `Latest agricultural census` (Năm thực hiện các khảo sát quy mô lớn).

**👉 Sheet `Series` (Từ điển Chỉ số - 20 trường):**
Giải thích "luật chơi" và ý nghĩa của hàng ngàn chỉ số.
* **Nhóm Định danh:** `Series Code` (Khóa ngoại liên kết sheet Data), `Indicator Name` (Tên chỉ số), `Topic` (Chủ đề phân loại: Y tế, Giáo dục, Kinh tế, Môi trường...).
* **Nhóm Định nghĩa:** `Short definition` (Định nghĩa tóm tắt), `Long definition` (Định nghĩa chi tiết, giải thích công thức tính), `Unit of measure` (Đơn vị tính).
* **Nhóm Phương pháp luận:** `Periodicity` (Chu kỳ thống kê, thường là Hàng năm), `Aggregation method` (Phương pháp cộng gộp dữ liệu), `Source` (Cơ quan gốc cung cấp số liệu, ví dụ: WHO, IMF, ILO).
* **Nhóm Lưu ý:** `Limitations and exceptions` (Những hạn chế hoặc sai số có thể có của chỉ số này), `Statistical concept and methodology` (Khái niệm thống kê chuyên sâu).

---
### 1.2. Các sheet phụ trợ (Auxiliary / Metadata Sheets)
Các sheet này cung cấp các chú thích (footnotes) giải thích sự bất thường hoặc thay đổi trong dữ liệu. Chữ `DESCRIPTION` trong các sheet này chứa nội dung văn bản giải thích.

* **👉 Sheet `country-series` (Quốc gia - Chỉ số):** Ghi chú các trường hợp ngoại lệ khi một chỉ số cụ thể được áp dụng cho một quốc gia cụ thể. *(Ví dụ: Giải thích tại sao chỉ số "Tỷ lệ thất nghiệp" của Việt Nam lại được tính theo một phương pháp hơi khác so với chuẩn chung của World Bank).*
* **👉 Sheet `series-time` (Chỉ số - Năm):** Ghi chú sự thay đổi về phương pháp thống kê hoặc định nghĩa của một chỉ số từ một năm cụ thể trở đi trên bình diện toàn cầu. *(Ví dụ: Từ năm 2011, World Bank thay đổi cách tính "Đường nghèo khổ" từ 1.25$ lên 1.9$/ngày).*
* **👉 Sheet `footnote` (Quốc gia - Chỉ số - Năm):** Ghi chú vi mô ở cấp độ một điểm dữ liệu duy nhất. *(Ví dụ: Giải thích tại sao số liệu dân số của Đức năm 1989 lại bị thiếu hoặc có sai số do sự kiện Bức tường Berlin).*

In [2]:
# Import các thư viện cần thiết
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown # Thư viện để in Markdown trong cell code

# Cài đặt hiển thị cho pandas
pd.set_option('display.max_columns', None)
import warnings
warnings.filterwarnings('ignore')

display(Markdown("> ✅ **Thành công:** Các thư viện đã được import. Bắt đầu đọc dữ liệu..."))

# Đường dẫn tới file Excel (Hãy đảm bảo file WDI nằm cùng thư mục)
file_path = '../../data/WDIEXCEL.xlsx'  # Thay đổi đường dẫn nếu cần

# Đọc các sheets
df_data = pd.read_excel(file_path, sheet_name='Data')
df_country = pd.read_excel(file_path, sheet_name='Country')
df_series = pd.read_excel(file_path, sheet_name='Series')

# Dùng Markdown để in ra cấu trúc thay vì dùng print
md_text = f"""
# 📊 2. Mô tả cấu trúc dữ liệu cơ bản
Sau khi nạp dữ liệu vào DataFrame, dưới đây là kích thước của 3 bảng cốt lõi:
- **Sheet 'Data'** : `{df_data.shape[0]:,}` bản ghi (dòng) x `{df_data.shape[1]}` trường dữ liệu (cột)
- **Sheet 'Country'**: `{df_country.shape[0]:,}` bản ghi (dòng) x `{df_country.shape[1]}` trường dữ liệu (cột)
- **Sheet 'Series'** : `{df_series.shape[0]:,}` bản ghi (dòng) x `{df_series.shape[1]}` trường dữ liệu (cột)
"""
display(Markdown(md_text))

> ✅ **Thành công:** Các thư viện đã được import. Bắt đầu đọc dữ liệu...


# 📊 2. Mô tả cấu trúc dữ liệu cơ bản
Sau khi nạp dữ liệu vào DataFrame, dưới đây là kích thước của 3 bảng cốt lõi:
- **Sheet 'Data'** : `403,256` bản ghi (dòng) x `70` trường dữ liệu (cột)
- **Sheet 'Country'**: `265` bản ghi (dòng) x `31` trường dữ liệu (cột)
- **Sheet 'Series'** : `1,516` bản ghi (dòng) x `20` trường dữ liệu (cột)


## 3. Tiền xử lý dữ liệu (Data Preprocessing)

Bảng `Data` hiện tại đang ở định dạng **Wide-format** (Mỗi năm là 1 cột). Định dạng này làm cho việc sử dụng các thư viện trực quan hóa (như Seaborn, Plotly) trở nên rất khó khăn. Chúng ta thực hiện các bước sau:
1. **Unpivot (Melt):** Gộp 66 cột năm (1960-2025) thành 2 cột `Year` và `Value` (Chuyển sang Long-format).
2. **Làm sạch:** Xóa các dòng có giá trị `Value` là `NaN` (giảm tải dung lượng rác).
3. **Data Enrichment (Làm giàu dữ liệu):** Ghép (Merge) các trường quan trọng như `Region`, `Income Group` (từ bảng Country) và `Topic` (từ bảng Series) vào bảng chính để phục vụ phân tích đa chiều.

In [5]:
# 1. Melt dữ liệu
id_vars = ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code']
years = [str(year) for year in range(1960, 2026) if str(year) in df_data.columns]

df_long = pd.melt(df_data, 
                  id_vars=id_vars, 
                  value_vars=years, 
                  var_name='Year', 
                  value_name='Value')

# Ép kiểu dữ liệu
df_long['Year'] = df_long['Year'].astype(int)

# 2. Loại bỏ các giá trị Null
df_clean = df_long.dropna(subset=['Value'])

# 3. Merge Metadata
df_final = df_clean.merge(df_country[['Country Code', 'Region', 'Income Group']], 
                          on='Country Code', 
                          how='left')

df_final = df_final.merge(df_series[['Series Code', 'Topic']], 
                          left_on='Indicator Code', 
                          right_on='Series Code', 
                          how='left')

df_final.drop(columns=['Series Code'], inplace=True)

# In kết quả bằng Markdown
display(Markdown(f"""
> ✅ **Hoàn tất Tiền xử lý:**
> * Kích thước bộ dữ liệu siêu chuẩn (Tidy Data) hiện tại: **`{df_final.shape[0]:,}` dòng** và **`{df_final.shape[1]}` cột**.
> * Xem thử 3 dòng đầu tiên của dữ liệu sau khi làm sạch:
"""))
display(df_final.head(3))
df_final.to_parquet('../../data/wdi_cleaned.parquet', index=False)


> ✅ **Hoàn tất Tiền xử lý:**
> * Kích thước bộ dữ liệu siêu chuẩn (Tidy Data) hiện tại: **`9,035,403` dòng** và **`9` cột**.
> * Xem thử 3 dòng đầu tiên của dữ liệu sau khi làm sạch:


,Country Name,Country Code,Indicator Name,Indicator Code,Year,Value,Region,Income Group,Topic
0,Africa Eastern and Southern,AFE,"Adolescent fertility rate (births per 1,000 wo...",SP.ADO.TFRT,1960,135.792572,NaN,NaN,Health: Reproductive health
1,Africa Eastern and Southern,AFE,Age dependency ratio (% of working-age populat...,SP.POP.DPND,1960,88.967790,NaN,NaN,Health: Population: Dynamics
2,Africa Eastern and Southern,AFE,"Age dependency ratio, old (% of working-age po...",SP.POP.DPND.OL,1960,5.631545,NaN,NaN,Health: Population: Dynamics


In [4]:
# Phân tích thống kê và in ra bằng Markdown

# 1. Thống kê theo Income Group
income_stats = df_final['Income Group'].value_counts(dropna=False).reset_index()
income_stats.columns = ['Nhóm Thu Nhập (Income Group)', 'Số lượng Record']

# 2. Top 5 chỉ số đầy đủ dữ liệu nhất
top_indicators = df_final['Indicator Name'].value_counts().head(5).reset_index()
top_indicators.columns = ['Tên Chỉ số (Indicator Name)', 'Số lượng điểm dữ liệu']

# 3. Thống kê mô tả cột số
desc_stats = df_final[['Year', 'Value']].describe().round(2)

# Tổng hợp hiển thị
display(Markdown("## 4. Phân tích Thống kê Cơ bản (Basic EDA)"))
display(Markdown("### 📌 4.1. Phân bố dữ liệu theo Nhóm thu nhập"))
display(income_stats)

display(Markdown("### 📌 4.2. Top 5 chỉ số được thu thập đầy đủ nhất trong lịch sử"))
display(top_indicators)

display(Markdown("""
### 📌 4.3. Thống kê phân phối (Descriptive Statistics)
*Lưu ý: Cột `Value` có độ lệch chuẩn (std) rất lớn vì chứa cả giá trị tỷ lệ (%) lẫn số liệu tuyệt đối lớn (như GDP hàng tỷ USD). Cần lọc theo từng `Indicator Name` khi phân tích sâu.*
"""))
display(desc_stats)

## 4. Phân tích Thống kê Cơ bản (Basic EDA)

### 📌 4.1. Phân bố dữ liệu theo Nhóm thu nhập

,Nhóm Thu Nhập (Income Group),Số lượng Record
0,High income,2642787
1,Upper middle income,1999166
2,Lower middle income,1914517
3,NaN,1598183
4,Low income,880750


### 📌 4.2. Top 5 chỉ số được thu thập đầy đủ nhất trong lịch sử

,Tên Chỉ số (Indicator Name),Số lượng điểm dữ liệu
0,Net migration,17490
1,"Population ages 80 and above, male (% of male ...",17225
2,"Population, male (% of total population)",17225
3,"Population, female (% of total population)",17225
4,"Population ages 75-79, male (% of male populat...",17225



### 📌 4.3. Thống kê phân phối (Descriptive Statistics)
*Lưu ý: Cột `Value` có độ lệch chuẩn (std) rất lớn vì chứa cả giá trị tỷ lệ (%) lẫn số liệu tuyệt đối lớn (như GDP hàng tỷ USD). Cần lọc theo từng `Indicator Name` khi phân tích sâu.*


,Year,Value
count,9035403.00,9.035403e+06
mean,2000.35,4.310685e+15
std,16.36,8.943650e+18
min,1960.00,-1.067341e+16
25%,1989.00,4.820000e+00
50%,2003.00,3.769000e+01
75%,2014.00,2.893805e+04
max,2025.00,2.435800e+22


## 5. Lộ trình Phân tích Dữ liệu (Data Storytelling)
Phần trực quan hóa dữ liệu (Data Visualization) sẽ được trình bày theo cấu trúc 4 chương, đi từ bức tranh toàn cảnh thế giới đến các phân tích chuyên sâu về khu vực Đông Nam Á.

### 📖 Chương 1: Vĩ mô & Sự phân hóa Kinh tế
*Mục đích: Thiết lập bối cảnh chung về sự bùng nổ của nhân loại và bóc tách sự bất bình đẳng giữa các nhóm quốc gia.*

* **1. Cú nhảy vọt của kinh tế toàn cầu**
    * **Câu hỏi:** Tổng sản phẩm quốc nội (GDP) của thế giới đã tăng trưởng như thế nào từ 1960? Các cuộc khủng hoảng lớn (2008, 2020) để lại "vết sẹo" nào trên biểu đồ?
    * **Chỉ số:** `GDP (current US$)` ở mức `World`.
    * **Biểu đồ đề xuất:** Biểu đồ đường (Line Chart) có đánh dấu (Annotate) các sự kiện lịch sử.
* **2. Dịch chuyển trọng tâm nhân khẩu học**
    * **Câu hỏi:** Dân số thế giới đang tăng chủ yếu ở đâu? Sự bùng nổ của Châu Á và Châu Phi diễn ra với tốc độ ra sao so với Châu Âu?
    * **Chỉ số:** `Population, total` nhóm theo `Region`.
    * **Biểu đồ đề xuất:** Biểu đồ miền xếp chồng (Stacked Area Chart).
* **3. Bức tường giàu nghèo (Wealth Gap)**
    * **Câu hỏi:** Nhóm các nước thu nhập cao (High Income) chiếm bao nhiêu phần trăm của cải (GDP) toàn cầu dù dân số của họ chỉ chiếm thiểu số?
    * **Chỉ số:** `GDP (current US$)` và `Population, total` nhóm theo `Income Group`.
    * **Biểu đồ đề xuất:** Biểu đồ bánh donut kép (Dual Donut Chart) hoặc Treemap so sánh tỷ trọng.

### 📖 Chương 2: Con người & Chất lượng Cuộc sống
*Mục đích: Đánh giá sự phát triển không chỉ qua tiền bạc, mà qua tuổi thọ, quá trình đô thị hóa và sự công bằng xã hội.*

* **4. Định lý Hans Rosling: Sức khỏe và Sự giàu có**
    * **Câu hỏi:** Quốc gia càng giàu thì người dân sống càng thọ, điều này có luôn đúng không? Nhóm nước nào đang tụt lại phía sau?
    * **Chỉ số:** Trục X: `GDP per capita`, Trục Y: `Life expectancy at birth`, Kích thước bọt: `Population`.
    * **Biểu đồ đề xuất:** Biểu đồ phân tán dạng bọt khí (Bubble Scatter Plot) với trục X dùng thang đo Logarit.
* **5. Cuộc đại di cư vào các thành phố**
    * **Câu hỏi:** Tốc độ đô thị hóa (người dân rời nông thôn lên thành thị) đang diễn ra nhanh nhất ở những khu vực nào trên thế giới?
    * **Chỉ số:** `Urban population (% of total population)` nhóm theo `Region`.
    * **Biểu đồ đề xuất:** Biểu đồ sườn dốc (Slope Chart) so sánh hai mốc thời gian (VD: 1990 và 2020).
* **6. Bình đẳng giới trong lực lượng lao động**
    * **Câu hỏi:** Tỷ lệ phụ nữ tham gia làm việc so với nam giới chênh lệch ra sao? Khu vực nào có sự bất bình đẳng giới lớn nhất?
    * **Chỉ số:** `Labor force participation rate, female` vs. `male`.
    * **Biểu đồ đề xuất:** Biểu đồ quả tạ (Dumbbell Plot) thể hiện khoảng cách (gap) nam - nữ.

### 📖 Chương 3: Cái giá của sự phát triển & Công nghệ
*Mục đích: Nhìn nhận những tác động phụ của tăng trưởng kinh tế (môi trường) và cách nhân loại dùng công nghệ để vượt qua.*

* **7. Dấu chân Carbon: Ai đang làm nóng trái đất?**
    * **Câu hỏi:** Top 20 quốc gia nào đang xả thải khí nhà kính (bình quân đầu người) cao nhất thế giới?
    * **Chỉ số:** `CO2 emissions (metric tons per capita)`.
    * **Biểu đồ đề xuất:** Biểu đồ cột ngang (Horizontal Bar Chart) kết hợp dải màu Heatmap.
* **8. Nỗ lực Xanh hóa (Energy Transition)**
    * **Câu hỏi:** Quá trình chuyển đổi sang năng lượng tái tạo diễn ra nhanh đến mức nào ở các nhóm thu nhập khác nhau?
    * **Chỉ số:** `Renewable energy consumption (% of total final energy consumption)`.
    * **Biểu đồ đề xuất:** Biểu đồ đường (Line Chart) so sánh xu hướng theo thời gian của các `Income Group`.
* **9. Kỷ nguyên kết nối toàn cầu**
    * **Câu hỏi:** Mạng Internet đã phủ sóng thế giới với tốc độ hàm mũ như thế nào kể từ đầu những năm 2000?
    * **Chỉ số:** `Individuals using the Internet (% of population)`.
    * **Biểu đồ đề xuất:** Đường cong chữ S (S-curve Line Chart).

### 📖 Chương 4: Tiêu điểm Khu vực - Đông Nam Á (ASEAN)
*Mục đích: Thu hẹp góc nhìn từ toàn cầu về một khu vực cụ thể, lấy Việt Nam làm trung tâm đối chiếu với các nước láng giềng.*

* **10. Cuộc đua GDP bình quân đầu người**
    * **Câu hỏi:** Sự vươn lên của Việt Nam so với các quốc gia ASEAN tiêu biểu (Thái Lan, Indonesia, Malaysia, Philippines) từ năm 1990 diễn ra như thế nào?
    * **Chỉ số:** `GDP per capita (constant 2015 US$)` lọc theo danh sách nước ASEAN.
    * **Biểu đồ đề xuất:** Biểu đồ đường nổi bật (Highlight Line Chart) - Tô đậm đường của Việt Nam (màu đỏ).
* **11. Thỏi nam châm thu hút FDI**
    * **Câu hỏi:** Việt Nam và các nước ASEAN đang thu hút dòng vốn đầu tư trực tiếp nước ngoài (FDI) mạnh mẽ ra sao qua các giai đoạn?
    * **Chỉ số:** `Foreign direct investment, net inflows (% of GDP)`.
    * **Biểu đồ đề xuất:** Biểu đồ cột nhóm (Grouped Bar Chart) so sánh theo từng giai đoạn 5 năm.
* **12. Ma trận Tương quan Phát triển tại ASEAN**
    * **Câu hỏi:** Tại khu vực Đông Nam Á, những yếu tố nào (y tế, giáo dục, đô thị hóa, internet) có sự tương quan mạnh mẽ nhất với thu nhập của người dân?
    * **Chỉ số:** Gom nhóm 8-10 chỉ số cốt lõi nhất ở năm gần nhất (VD: 2022).
    * **Biểu đồ đề xuất:** Biểu đồ nhiệt tương quan (Correlation Heatmap) để tóm lược và kết thúc báo cáo.